# CP5 — Final Model Export & Smoke Test
**Thesis:** An Explainable ML Analysis of Race Outcome Determinants in the 2022–2025 F1 Ground-Effect Era.

Checkpoint 5 of 5 — the final ML notebook. Trains both **Random Forest** models on 2022–2024, persists
every artifact a platform needs, writes a model card, and runs a load-from-disk smoke test. **No new
modelling or tuning** — these are the exact CP3/CP3.1 models, re-fit and saved.

### Decisions / deviations (verified against CP2–CP4 outputs)
- **`pit_lane_start` does not exist.** The CP5 spec's `PRE_RACE_FEATURES` lists a `pit_lane_start` column;
  in CP2 we instead remapped `grid_position 0 → 21` (pit-lane = behind P20), so that signal lives inside
  `grid_position`. Saving the spec's list verbatim would break the platform with a `KeyError`. We save the
  **actual trained 13-feature list** from `models/pre_race_feature_list.json` (its 13th feature is
  `total_race_laps`). This is the list Model A was trained on in CP3.1.
- **Reuse the fitted imputers** (`imputer.pkl`, `pre_race_imputer.pkl`) rather than refit, so each saved
  model is a **matched (model, imputer) pair** for the platform: raw input → impute → predict.
- **Both models are Random Forest** (selected in CP3 and CP3.1). MAE sanity check must reproduce
  ~2.545 (B) and ~3.332 (A) within 0.05, else stop.

## Setup — load feature lists, fitted imputers, encoders, and data

In [ ]:
from pathlib import Path
import json, pickle, datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
TARGET = "final_position"

# Feature lists exactly as CP3/CP3.1 trained them (source of truth on disk).
FULL_FEATURES = json.load(open(MODELS_DIR / "feature_list.json"))             # Model B (24)
PRE_RACE_FEATURES = json.load(open(MODELS_DIR / "pre_race_feature_list.json"))  # Model A (13; total_race_laps, NOT pit_lane_start)
assert "pit_lane_start" not in PRE_RACE_FEATURES, "remap convention changed unexpectedly"
print("Model B features:", len(FULL_FEATURES), "| Model A features:", len(PRE_RACE_FEATURES))

# Fitted artifacts from CP3/CP3.1 (do NOT refit -> keeps (model, imputer) pairs consistent).
imputer_b = pickle.load(open(MODELS_DIR / "imputer.pkl", "rb"))
imputer_a = pickle.load(open(MODELS_DIR / "pre_race_imputer.pkl", "rb"))
encoders  = pickle.load(open(MODELS_DIR / "encoders.pkl", "rb"))
print("Loaded fitted imputer.pkl, pre_race_imputer.pkl, encoders.pkl")

# Data
train_b = pd.read_csv(PROC / "train.csv");                  test_b = pd.read_csv(PROC / "test.csv")
train_a = pd.read_csv(PROC / "pre_race" / "pre_race_train.csv"); test_a = pd.read_csv(PROC / "pre_race" / "pre_race_test.csv")
print("Train/test  B:", train_b.shape, test_b.shape, "| A:", train_a.shape, test_a.shape)

## Step 1 — Train both final models (2022–2024) and verify MAE
Same known-issue handling as CP3 (rookie codes → median **train** code; `constructor_season_points`
NaN → 0), then impute with the loaded train-fit imputers and fit Random Forest. The reproduced test
MAEs must match CP3/CP3.1 within 0.05.

In [ ]:
rf_params = dict(n_estimators=500, max_depth=None, min_samples_leaf=2,
                 max_features="sqrt", random_state=42, n_jobs=-1)

def remap_rookies(train_df, test_df, col):
    known = set(train_df[col].unique())
    med = int(np.round(train_df[col].median()))
    test_df[col] = test_df[col].where(test_df[col].isin(known), med)

for tr, te in [(train_b, test_b), (train_a, test_a)]:
    remap_rookies(tr, te, "driver_encoded")
    remap_rookies(tr, te, "team_encoded")
    for d in (tr, te):
        d["constructor_season_points"] = d["constructor_season_points"].fillna(0.0)

# Build matrices, impute with the loaded (train-fit) imputers, fit RF.
y_train = train_b[TARGET].values
Xtr_b = pd.DataFrame(imputer_b.transform(train_b[FULL_FEATURES]),     columns=FULL_FEATURES)
Xte_b = pd.DataFrame(imputer_b.transform(test_b[FULL_FEATURES]),      columns=FULL_FEATURES)
Xtr_a = pd.DataFrame(imputer_a.transform(train_a[PRE_RACE_FEATURES]), columns=PRE_RACE_FEATURES)
Xte_a = pd.DataFrame(imputer_a.transform(test_a[PRE_RACE_FEATURES]),  columns=PRE_RACE_FEATURES)

model_b = RandomForestRegressor(**rf_params).fit(Xtr_b, y_train)
model_a = RandomForestRegressor(**rf_params).fit(Xtr_a, train_a[TARGET].values)

y_test = test_b[TARGET]                     # pandas Series (smoke test uses .iloc[0])
mae_a = mean_absolute_error(test_a[TARGET].values, model_a.predict(Xte_a))
mae_b = mean_absolute_error(y_test.values,         model_b.predict(Xte_b))
print(f"Model A (pre-race) test MAE: {mae_a:.3f}  (expected ~3.332)")
print(f"Model B (full)     test MAE: {mae_b:.3f}  (expected ~2.545)")
assert abs(mae_a - 3.332) <= 0.05, f"STOP: Model A MAE off by >0.05 ({mae_a:.3f})"
assert abs(mae_b - 2.545) <= 0.05, f"STOP: Model B MAE off by >0.05 ({mae_b:.3f})"
print("MAE sanity check passed (both within 0.05).")

## Step 2 — Save model files

In [ ]:
with open(MODELS_DIR / "model_a.pkl", "wb") as f:
    pickle.dump(model_a, f)
with open(MODELS_DIR / "model_b.pkl", "wb") as f:
    pickle.dump(model_b, f)

# Per-model feature lists (the ACTUAL trained lists; A has total_race_laps, not pit_lane_start).
with open(MODELS_DIR / "feature_list_a.json", "w") as f:
    json.dump(PRE_RACE_FEATURES, f, indent=2)
with open(MODELS_DIR / "feature_list_b.json", "w") as f:
    json.dump(FULL_FEATURES, f, indent=2)
print("Saved: model_a.pkl, model_b.pkl, feature_list_a.json, feature_list_b.json")

# Verify the pre-existing artifacts the platform also needs are present.
for fname in ["feature_list.json", "imputer.pkl", "pre_race_imputer.pkl", "encoders.pkl"]:
    assert (MODELS_DIR / fname).exists(), f"Missing: models/{fname}"
    print(f"  ✓ models/{fname}")

## Step 3 — Write model_card.json
Metrics are computed live (not hardcoded) so the card cannot drift from the saved models.

In [ ]:
def metrics(y_true, pred):
    yr = np.round(pred)
    return {"test_mae": round(float(mean_absolute_error(y_true, pred)), 3),
            "test_r2": round(float(r2_score(y_true, pred)), 3),
            "spearman_r": round(float(spearmanr(y_true, pred).correlation), 3),
            "within_3": round(float(np.mean(np.abs(yr - y_true) <= 3)), 3),
            "within_5": round(float(np.mean(np.abs(yr - y_true) <= 5)), 3)}

ma = metrics(test_a[TARGET].values, model_a.predict(Xte_a))
mb = metrics(y_test.values,         model_b.predict(Xte_b))
base_grid = round(float(mean_absolute_error(y_test.values, test_b["grid_position"].values)), 3)
base_mean = round(float(mean_absolute_error(y_test.values, np.full(len(y_test), y_train.mean()))), 3)

model_card = {
    "trained_on": "2022-2024",
    "evaluated_on": "2025",
    "train_date": datetime.date.today().isoformat(),
    "model_type": "Random Forest Regressor (n=500)",
    "rf_params": rf_params,
    "model_a": {"name": "Pre-race Forecasting Model", "pickle": "model_a.pkl",
                "feature_list": "feature_list_a.json", "imputer": "pre_race_imputer.pkl",
                "features": len(PRE_RACE_FEATURES), **ma},
    "model_b": {"name": "Full Race Reconstruction Model", "pickle": "model_b.pkl",
                "feature_list": "feature_list_b.json", "imputer": "imputer.pkl",
                "features": len(FULL_FEATURES), **mb},
    "baselines": {"grid_position_mae": base_grid, "mean_prediction_mae": base_mean},
    "inference_protocol": "raw row -> select feature_list -> imputer.transform -> model.predict",
    "notes": [
        "Model A uses only pre-race features — true forecasting (does not reliably beat the grid baseline).",
        "Model B uses in-race features — race reconstruction and SHAP explainability.",
        "driver_encoded and team_encoded are label integers from encoders.pkl (partly memorisation).",
        "2025 rookies (ANT, BOR, HAD) map to the median training driver code at debut.",
        "is_wet_race in Model A is a pre-race weather-forecast proxy assumption.",
        "Pit-lane starts are encoded as grid_position=21 (no separate pit_lane_start feature)."
    ]
}
with open(MODELS_DIR / "model_card.json", "w") as f:
    json.dump(model_card, f, indent=2)
print("Saved models/model_card.json")
print(json.dumps({"model_a": model_card["model_a"], "model_b": model_card["model_b"]}, indent=2))

## Step 4 — Smoke test (load everything from disk, predict one row)
Mirrors the platform's inference path: load the pickled model + its imputer + feature list, then predict
a single raw test row through impute → predict.

In [ ]:
loaded_model    = pickle.load(open(MODELS_DIR / "model_b.pkl", "rb"))
loaded_imputer  = pickle.load(open(MODELS_DIR / "imputer.pkl", "rb"))
loaded_encoders = pickle.load(open(MODELS_DIR / "encoders.pkl", "rb"))
loaded_features = json.load(open(MODELS_DIR / "feature_list_b.json"))

# Raw test row -> impute with the loaded imputer -> predict (exactly the platform flow).
raw_row = test_b[loaded_features].iloc[0:1]
X_smoke = pd.DataFrame(loaded_imputer.transform(raw_row), columns=loaded_features)
pred = loaded_model.predict(X_smoke)[0]
actual = y_test.iloc[0]
print(f"Smoke test — Driver: {test_b['driver'].iloc[0]} @ {test_b['event_name'].iloc[0]}")
print(f"Predicted: P{pred:.1f} | Actual: P{actual:.0f} | Error: {abs(pred - actual):.1f}")
smoke_ok = abs(pred - actual) < 10
print("✓ Passed" if smoke_ok else "✗ Check results")

## CP5 Summary

In [ ]:
print("=== CP5 COMPLETE ===")
print("MAE check:")
print(f"  Model A: {mae_a:.3f} (expected ~3.332)")
print(f"  Model B: {mae_b:.3f} (expected ~2.545)")
print("Files saved:")
for f in ["model_a.pkl", "model_b.pkl", "feature_list_a.json", "feature_list_b.json", "model_card.json"]:
    print(f"  ✓ {f}")
print("Files verified (existing):")
for f in ["feature_list.json", "imputer.pkl", "pre_race_imputer.pkl", "encoders.pkl"]:
    print(f"  ✓ {f}")
print(f"Smoke test: Predicted P{pred:.1f} | Actual P{actual:.0f} | {'✓ passed' if smoke_ok else '✗ check'}")
print("ML pipeline complete (CP1 EDA -> CP2 features -> CP3/3.1 train -> CP4 SHAP -> CP5 export).")